# U1-L1 实验：K1 连接与信息探索**实验目标**连接 K1 机器人，通过 BoosterOS SDK 接口读取并打印：- 机器人的身份信息（制造商 / 型号 / 序列号 / 固件版本）- 关节清单（数量、名称、限位、最大速度）- 预定义动作库（ID / 类型 / 时长 / 可中断）- 当前运行状态（模式、电量）> ⚠️ **安全提示**：本实验全部为**只读操作**，不涉及任何机器人运动，可放心运行。运行前请确认：> - ① K1 已开机并处于 STAND 状态；> - ② 电脑与 K1 在同一 WiFi；> - ③ 已 `pip install boosteros`。> 📌 **前置准备**：本实验需在 K1 已完成**开机、配网（连接同一 WiFi）、查看 IP**，并能用 **BoosterStudio** 成功连接机器人之后进行。未完成上述准备，BoosterOS SDK 程序无法运行。

## 步骤 1：连接 K1 机器人

`BoosterRobot` 是操作 K1 的统一入口。运行下面单元格建立连接。

若连接失败，请检查：K1 是否开机、电脑与 K1 是否同一 WiFi、SDK 是否已安装。
- 如果提示 `ModuleNotFoundError: No module named 'boosteros'`，代表 boosteros 没有安装

In [ ]:
from boosteros.robots.booster import BoosterRobot

robot = BoosterRobot()
print("已连接 K1，开始探索 🤖")

## 步骤 2：读取机器人身份信息

`robot.robot_info` 返回一个只读的元信息对象，包含制造商、型号、序列号、固件版本等。认识一台新机器人的第一步就是看清它的“身份证”。

In [ ]:
info = robot.robot_info
print(f"机器人的元信息：{info}")

`RobotInfo` 表示机器人的基础信息，具体包括：

| 属性　　　　　　 | 类型　　　　　　　| 说明　　　　　　　　　　　　　 |
| ------------------| -------------------| --------------------------------|
| manufacturer　　 | str　　　　　　　 | 机器人厂家名称　　　　　　　　 |
| model　　　　　　| str　　　　　　　 | 机器人型号；不可用时为空字符串 |
| name　　　　　　 | str　　　　　　　 | 机器人名称；不可用时为空字符串 |
| serial_number　　| str　　　　　　　 | 序列号；不可用时为空字符串　　 |
| firmware_version | str　　　　　　　 | 固件版本号；不可用时为空字符串 |
| extra　　　　　　| dict[str, object] | 扩展信息　　　　　　　　　　　 |

**为什么要专门读取这些信息？**

- **型号（model）**：决定机器人的硬件规格与能力边界。不同型号支持的接口、动作、参数范围不同，程序必须针对具体机型编写与调试。
- **序列号（serial_number）**：机器人的唯一身份标识，相当于“身份证号”。在多台机器人并存的教学或实训环境中，凭序列号可区分、定位具体设备。
- **固件版本（firmware_version）**：机器人底层控制软件的版本号。不同固件版本下，接口行为、动作库、参数范围可能存在差异。运行 SDK 程序前确认固件版本，可避免因版本不匹配导致的异常，也便于统一实验环境、稳定复现结果。

In [ ]:
print("型号：    ", info.model)
print("序列号：  ", info.serial_number)
print("固件版本：", info.firmware_version)

## 步骤 3：浏览关节清单

`robot.list_joints()` 返回所有关节的信息，可以打印查看。

In [ ]:
joints = robot.list_joints()
print(f"关节总数：{len(joints)}")
print(f"list_joints{joints}")

可以发现 K1 机器人的 `robot.list_joints()` 中包含 22 个关节的数据，其中每一个关节又包括：



| 属性　　　　 | 类型　　　　　　　| 说明　　　　　　　　　　　　　　　　|

| --------------| -------------------| -------------------------------------|

| name　　　　 | str　　　　　　　 | 关节名称　　　　　　　　　　　　　　|

| limits　　　 | JointLimits　　　 | 关节位置范围（min / max，单位 rad） |

| max_torque　 | float　　　　　　 | 关节最大力矩　　　　　　　　　　　　|

| max_velocity | float　　　　　　 | 关节最大速度（单位 rad/s）　　　　　|

| extra　　　　| dict[str, object] | 扩展信息　　　　　　　　　|



观察关节的**数量与命名规律**（头部、手臂、腿部等），即可直观理解 K1 的“身体结构”。

In [ ]:
for j in joints:
    print(f"{j.name:<20} 限位: [{j.limits.min:.3f}, {j.limits.max:.3f}] rad  最大速度: {j.max_velocity:.3f} rad/s")

## 步骤 4：浏览预定义动作库

`robot.list_actions()` 返回出厂预置的动作列表（比如挥手、鞠躬、踢球等）。

In [ ]:
actions = robot.list_actions()
print(f"预定义动作总数：{len(actions)}")
print(f"预定义动作内容：{actions}")

可以发现 K1 机器人包含多个预定义动作，具体每个动作又包括：

| 属性 | 类型 | 说明 |
|---|---|---|
| id | str | 动作 ID，可作为 `robot.do_action(action_id)` 的 action_id 参数 |
| type | str | 动作类型，如 "upper_body"、"whole_body" |
| duration | float | 预估持续时间，`-1` 表示不确定 |
| interruptible | bool | 是否可被新指令打断 |

In [ ]:
for a in actions:
    dur = f"{a.duration:.1f}s" if a.duration is not None else "-"
    print(f"动作名称：{a.id:<25} 类型：{a.type:<15} 时长：{dur:<8} 可中断：{'是' if a.interruptible else '否'}")

## 步骤 5：查看当前状态

机器人不仅有身体结构，还持续维护着自身的运行状态。`get_mode()` 返回机器人当前运行模式（如待机 STAND、运动等）；`get_battery()` 返回电量信息。这两项是我们最先能直接读取的“机器人生理指标”。

In [ ]:
print("当前模式：", robot.get_mode())
print("电池信息：", robot.get_battery())

## 步骤 6（选做）：关节实时角度

`get_joint_states()` 可读取关节当前实时角度（`position`，单位 rad）。

提示：取消注释即可运行程序。

In [ ]:
# 取消下面注释即可运行
# states = robot.get_joint_states()
# for name in states.names[:10]:
#     print(name, states.get_joint(name).position)

## 学习评价

请独立完成以下题目，检验本实验的掌握情况。

**一、判断题**（对的打√，错的打×）

1. 本实验全部为只读操作，不涉及机器人任何运动，可以放心运行。（    ）

**二、填空题**

2. `robot.robot_info` 返回的对象中，相当于机器人“身份证号”、用于多机环境中区分与定位设备的字段是 ______。

**三、选择题**（单选）

3. 在 `robot.list_actions()` 返回的动作信息中，可作为 `robot.do_action()` 的参数来指定要执行哪个动作的字段是（    ）。
   A. `type`　　　B. `id`　　　C. `duration`　　　D. `interruptible`

> 参考答案（教师留存，勿印发给学生）：
> 1. √　2. `serial_number`（序列号）　3. B